In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml
from gluonts.dataset.util import to_pandas

from tsfm_lens.dataset import GiftEvalDataset
from tsfm_lens.utils import (
    apply_custom_style,
)

In [ ]:
apply_custom_style("../../config/plotting.yaml")

In [ ]:
DEFAULT_COLORS = list(plt.get_cmap("tab20").colors)  # type: ignore
print(f"{len(DEFAULT_COLORS)} colors")

In [ ]:
WORK_DIR = os.getenv("WORK", "/work")
DATA_DIR = os.path.join(WORK_DIR, "data")
plot_save_dir = os.path.join("../../figs", "dataset_viz")
os.makedirs(plot_save_dir, exist_ok=True)

In [ ]:
split_dir = os.path.join(DATA_DIR, "gift-eval")
system_name = "m4_monthly"
term = "long"

In [ ]:
dataset = GiftEvalDataset(name=system_name, term=term, data_dir=split_dir)
print(f"data dimension: {dataset.target_dim}")
print(f"term: {dataset.term}")
print(f"prediction length: {dataset.prediction_length}")
print(f"min series length: {dataset._min_series_length}")
print(f"num windows: {dataset.windows}")
print(f"num rows: {dataset.hf_dataset.num_rows}")

# Get first test sample
context_entry = next(iter(dataset.test_data.input))
future_entry = next(iter(dataset.test_data.label))
context_target = context_entry["target"]
future_target = future_entry["target"]

# Select dimensions to plot
max_dims_to_plot = 10
num_dims_to_plot = min(dataset.target_dim, max_dims_to_plot)
selected_dims = np.arange(num_dims_to_plot)
context_target = context_target[selected_dims] if dataset.target_dim > 1 else context_target[None, ...]
future_target = future_target[selected_dims] if dataset.target_dim > 1 else future_target[None, ...]

# Plot each dimension
fig, axes = plt.subplots(num_dims_to_plot, 1, figsize=(5, 2 * num_dims_to_plot))
axes = [axes] if num_dims_to_plot == 1 else axes

for dim, ax in enumerate(axes):
    context_series = to_pandas({"target": context_target[dim], "start": context_entry["start"]})
    future_series = to_pandas({"target": future_target[dim], "start": future_entry["start"]})

    context_series.plot(color="black", linewidth=1, ax=ax, label="Context")
    future_series.plot(color="tab:blue", linewidth=1, ax=ax, label="Ground Truth")
    ax.grid(which="both")
    ax.set_title(f"Dimension {dim}")
    ax.legend()

plt.tight_layout()
plt.show()

# Check for NaN values and convert to tensors
num_nans_context = sum(
    to_pandas({"target": context_target[dim], "start": context_entry["start"]}).isna().sum()
    for dim in range(num_dims_to_plot)
)
num_nans_future = sum(
    to_pandas({"target": future_target[dim], "start": future_entry["start"]}).isna().sum()
    for dim in range(num_dims_to_plot)
)

context = torch.tensor(context_target)
future_vals = torch.tensor(future_target)

print(f"num nans context: {num_nans_context}")
print(f"num nans future vals: {num_nans_future}")
print(f"context shape: {context.shape}")
print(f"future vals shape: {future_vals.shape}")

### Examine All Datasets of a Term

In [ ]:
term

In [ ]:
evaluation_yaml_path = os.path.join("../../config", "evaluation.yaml")
with open(evaluation_yaml_path) as f:
    gift_eval_config = yaml.safe_load(f)["eval"]["gift_eval"]


In [ ]:
short_term_datasets = gift_eval_config["short_datasets"]
print(f"{len(short_term_datasets)} short term datasets: {short_term_datasets}")

In [ ]:
med_long_term_datasets = gift_eval_config["medium_long_datasets"]
print(f"{len(med_long_term_datasets)} med long term datasets: {med_long_term_datasets}")


In [ ]:
all_datasets = {"short": short_term_datasets, "long": med_long_term_datasets}

Note: "Rows" is number of series, and "Min Len" is minimum value of (context + prediction) length among all splits of that dataset

In [ ]:
print("Long Term Datasets")
selected_datasets = all_datasets["long"]
print(f"{'Dataset Name':<30} {'Target Dim':>10} {'Pred Len':>10} {'Min Len':>10} {'Windows':>10} {'Rows':>10}")
print("-" * 80)
for dataset_name in selected_datasets:
    dataset = GiftEvalDataset(name=dataset_name, term=term, data_dir=split_dir)
    print(
        f"{dataset_name:<30} {dataset.target_dim:>10} {dataset.prediction_length:>10} {dataset._min_series_length:>10} {dataset.windows:>10} {dataset.hf_dataset.num_rows:>10}"
    )

In [ ]:
print("Short Term Datasets")
selected_datasets = all_datasets["short"]
print(f"{'Dataset Name':<30} {'Target Dim':>10} {'Pred Len':>10} {'Min Len':>10} {'Windows':>10} {'Rows':>10}")
print("-" * 80)
for dataset_name in selected_datasets:
    dataset = GiftEvalDataset(name=dataset_name, term=term, data_dir=split_dir)
    print(
        f"{dataset_name:<30} {dataset.target_dim:>10} {dataset.prediction_length:>10} {dataset._min_series_length:>10} {dataset.windows:>10} {dataset.hf_dataset.num_rows:>10}"
    )

In [ ]:
dataset = GiftEvalDataset(name="electricity/15T", term="short", data_dir=split_dir)


In [ ]:
len(dataset.test_data)